# 01 | Single-cell data retrieval from CellProfiler SQLite databases

**Purpose.** Merge object-level measurements from three independent CellProfiler `ExportToDatabase` plate databases. Retain nuclei, cytoplasm, and whole-cell measurements, and export a consolidated single-cell table.

**Input:** one SQLite database per plate, in the directory layout expected by the Broad Institute `generate_profiles.merge_sqlite_dataframes` function. **Output:** `AuNP_rato_serum_time_Walker_SingleCells.csv`.

**External dependency:** [Broad Institute scripts_notebooks_fossa](https://github.com/broadinstitute/scripts_notebooks_fossa). The original analysis used this repository; install or clone it before running.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Edit only this path to point to the parent folder of scripts_notebooks_fossa.
SCRIPTS_PARENT = Path("../external").resolve()
sys.path.insert(0, str(SCRIPTS_PARENT))
from scripts_notebooks_fossa.profiles import generate_profiles

# Parent directory containing the individual plate folders.
DATABASE_ROOT = Path("../data/sqlite").resolve()
OUTPUT_DIR = Path("../results").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Define experimental plates and segmented compartments

The three plate identifiers are retained exactly as recorded in the original experiment. For each plate, CellProfiler object measurements from nuclei, cytoplasm, and whole cells are retrieved. The database layout and plate-name matching must conform to the upstream helper function.

In [ ]:
PLATE_IDS = [
    "250718_150938_Plate_1_",
    "250725_152547_Plate_1_",
    "250730_153934_Plate_1_",
]
COMPARTMENTS = ["Nuclei", "Cytoplasm", "Cells"]
print(f"Plates: {len(PLATE_IDS)}; compartments: {', '.join(COMPARTMENTS)}")

## 2. Merge object measurements from SQLite

`merge_sqlite_dataframes` uses the original CellProfiler-export database structure to associate measurements from the specified compartments. The resulting plate-level DataFrames are checked before concatenation.

In [ ]:
if not DATABASE_ROOT.is_dir():
    raise FileNotFoundError(f"Set DATABASE_ROOT to the folder containing the plate databases: {DATABASE_ROOT}")
plate_tables = generate_profiles.merge_sqlite_dataframes(
    PLATE_IDS, str(DATABASE_ROOT), COMPARTMENTS
)
if not plate_tables:
    raise ValueError("No plate tables were returned; verify the plate names and database layout.")
for plate, table in zip(PLATE_IDS, plate_tables):
    print(f"{plate}: {table.shape[0]:,} rows × {table.shape[1]:,} columns")

## 3. Combine plates and export the single-cell profile

Rows from the three plates are concatenated without averaging or normalizing the measurements. The row index is reset and excluded from the exported CSV. This table is the input for Notebook 02.

In [ ]:
single_cells = pd.concat(plate_tables, axis=0, ignore_index=True)
output_file = OUTPUT_DIR / "AuNP_rato_serum_time_Walker_SingleCells.csv"
single_cells.to_csv(output_file, index=False)
print(f"Saved {single_cells.shape[0]:,} rows × {single_cells.shape[1]:,} columns to {output_file}")